# TP Individual — QuéLibroLeo
### E72.1.01 · Fundamentos de Métodos Analíticos Predictivos

**Problema.** Predecir si a un lector le va a gustar un libro que no leyó, a partir de
los datos de quelibroleo.com. Es una **clasificación binaria**: `rating >= 7` es que le
gustó (`1`), `rating <= 5` que no (`0`), y los `rating == 6` se descartan por ser un
valor gris que no aporta información.

**Métrica de decisión: `f1`.** El dataset está desbalanceado (~84/16), así que `accuracy`
no sirve para decidir nada. Todas las comparaciones entre experimentos usan `f1` y sólo
`f1`; la matriz de confusión se muestra como material descriptivo.

**Semilla: 42** en todo (split, modelos, samplers, muestreos), para que los errores sean
comparables entre corridas.

> Este notebook se genera automáticamente desde los módulos de `src/` con
> `python -m src.armar_notebook`. No editarlo a mano: los cambios se hacen en los `.py`.


## 0. Los datos

Los tres CSV no están en el repositorio. En Colab, montá el Drive y apuntá
`QLL_DATA_DIR` a la carpeta que los contiene. Si corrés local con los CSV al lado del
notebook, no hace falta tocar nada.


In [1]:
import os

# --- Colab: descomentar estas dos líneas y ajustar la ruta ---
# from google.colab import drive; drive.mount("/content/drive")
# os.environ["QLL_DATA_DIR"] = "/content/drive/MyDrive/TP_QueLibroLeo/data"

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)


## 1. Configuración

Configuración global del TP: rutas, semilla, métrica de decisión y cortes de rating.

Todo lo que otro módulo necesite parametrizar vive acá. Nadie más hardcodea rutas
ni números mágicos.

*(desde `src/config.py`)*


In [2]:
import os
from pathlib import Path

# --------------------------------------------------------------------------------------
# Rutas
# --------------------------------------------------------------------------------------

def _raiz() -> Path:
    """Raíz del proyecto, tanto corriendo como módulo como dentro del notebook.

    En el notebook no existe `__file__` y el directorio de trabajo puede ser
    `notebooks/`, así que se sube hasta encontrar el CLAUDE.md. En Colab, donde no
    está, cae en el directorio de trabajo y todo cuelga de ahí.
    """
    if "__file__" in globals():
        return Path(__file__).resolve().parent.parent
    actual = Path.cwd().resolve()
    for candidata in (actual, *actual.parents):
        if (candidata / "CLAUDE.md").is_file():
            return candidata
    return actual


RAIZ = _raiz()

# Los CSV no están versionados. Por defecto se buscan en data/; se puede apuntar a
# otro lado con la variable de entorno QLL_DATA_DIR (útil en Colab, donde los datos
# quedan montados en Drive).
DIR_DATOS = Path(os.environ.get("QLL_DATA_DIR", RAIZ / "data"))
DIR_CHECKPOINTS = Path(os.environ.get("QLL_CHECKPOINTS_DIR", RAIZ / "checkpoints"))
DIR_RESULTADOS = RAIZ / "resultados"
DIR_FIGURAS = DIR_RESULTADOS / "figuras"

# Nombres de archivo tal como los entregó la cátedra. La tabla de opiniones viene
# con el nombre "interacciones.csv".
ARCHIVO_LIBROS = os.environ.get("QLL_CSV_LIBROS", "libros.csv")
ARCHIVO_LECTORES = os.environ.get("QLL_CSV_LECTORES", "lectores.csv")
ARCHIVO_OPINIONES = os.environ.get("QLL_CSV_OPINIONES", "interacciones.csv")


def resolver_csv(nombre: str) -> Path:
    """Devuelve la ruta del CSV: primero en DIR_DATOS, si no está, en la raíz del repo.

    El fallback a la raíz existe porque los CSV de la cátedra hoy están ahí; la ruta
    canónica sigue siendo data/. Si no aparece en ninguna de las dos, devuelve la
    canónica para que el error de lectura indique dónde había que ponerlo.
    """
    for candidata in (DIR_DATOS / nombre, RAIZ / nombre):
        if candidata.is_file():
            return candidata
    return DIR_DATOS / nombre


CSV_LIBROS = resolver_csv(ARCHIVO_LIBROS)
CSV_LECTORES = resolver_csv(ARCHIVO_LECTORES)
CSV_OPINIONES = resolver_csv(ARCHIVO_OPINIONES)

CHECKPOINT_BASE = DIR_CHECKPOINTS / "01_base.pkl"
CHECKPOINT_LIMPIO = DIR_CHECKPOINTS / "04_limpio.pkl"

# --------------------------------------------------------------------------------------
# Reproducibilidad y evaluación
# --------------------------------------------------------------------------------------

SEED = 42

# Métrica única de decisión (CLAUDE.md 3.2). El dataset está desbalanceado ~80/20,
# así que accuracy está prohibida. Todas las comparaciones entre experimentos se
# hacen con esta y sólo con esta.
METRICA = "f1"

# Sobre qué clase se calcula la métrica. Se mide la 0 ("no le gustó"), que es la
# minoritaria y la informativa. Con la clase 1, predecir siempre "le gustó" da
# f1 = 0.9123 sin aprender nada, y ningún modelo razonable lo supera: la métrica
# quedaría dominada por la proporción de clases en vez de medir señal. Sobre la
# clase 0 ese clasificador trivial da f1 = 0, así que toda mejora es real.
CLASE_MEDIDA = 0

# Proporción del test en el split. El split es estratificado porque las clases
# están ~84/16 y un split al azar podría desbalancearlas todavía más.
TEST_SIZE = 0.25

# Hiperparámetros del modelo congelado de evaluación (CLAUDE.md 3.4). No se tocan
# hasta la rama de optimización: si el modelo cambia entre experimentos, la
# comparación entre ellos no mide el cambio, mide el modelo.
MODELO_CONGELADO = {
    "n_estimators": 300,
    "max_depth": 8,
    "class_weight": "balanced",
    "random_state": SEED,
    "n_jobs": -1,
}

# Valor con el que se rellenan los nulos que sobrevivan, sólo para que el modelo
# pueda correr sobre un dataset sucio. Es un centinela, no una imputación: imputar
# es una decisión de la rama de limpieza y el instrumento de medición no la toma
# por su cuenta.
RELLENO_NULOS = -1

TABLA_EXPERIMENTOS = DIR_RESULTADOS / "tabla_experimentos.csv"

# Banda de ruido del instrumento, medida: el mismo dataset evaluado con 6 semillas de
# split distintas da σ = 0.0024, así que 2σ ≈ 0.005. Un delta más chico que esto no se
# distingue del azar del split y no alcanza para declarar que un cambio sirve.
UMBRAL_RUIDO = 0.005

# --------------------------------------------------------------------------------------
# Definición del target
# --------------------------------------------------------------------------------------

COL_RATING = "rating"
TARGET = "gusto"

RATING_MIN_GUSTO = 7      # rating >= 7  -> gusto = 1
RATING_MAX_NO_GUSTO = 5   # rating <= 5  -> gusto = 0
RATING_DESCARTADO = 6     # rating == 6  -> la fila se elimina (rating gris)

# --------------------------------------------------------------------------------------
# Unión de tablas
# --------------------------------------------------------------------------------------

COL_ID_LIBRO = "id_libro"
COL_ID_LECTOR = "id_lector"

# Cómo se resuelven las opiniones que apuntan a un libro o a un lector inexistente.
# Se decide con el diagnóstico de src/diagnostico.py.
HOW_UNION = "left"


En los `.py` el código está repartido en módulos y se referencia como `config.SEED` o
`carga.unir(...)`. En el notebook es todo un mismo espacio de nombres, así que estos alias
hacen que esas referencias sigan funcionando sin tener que reescribir una sola línea.


In [3]:
import sys

_yo = sys.modules["__main__"]
config = carga = diagnostico = eda = limpieza = modelo = experimentos = _yo


## 2. Carga y unión de las tres tablas

Lectura y unión de las tres tablas. Acá no se limpia nada: sólo se lee, se une
y se construye el target.

*(desde `src/carga.py`)*


In [4]:
import pandas as pd



# --------------------------------------------------------------------------------------
# Lectura
# --------------------------------------------------------------------------------------

def cargar_tablas() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Lee los tres CSV tal cual vienen y los devuelve por separado, sin transformar.

    No se castean tipos ni se parsean fechas a propósito: cualquier conversión es una
    decisión de limpieza y va en la rama que corresponde.
    """
    libros = pd.read_csv(config.CSV_LIBROS, low_memory=False)
    lectores = pd.read_csv(config.CSV_LECTORES, low_memory=False)
    opiniones = pd.read_csv(config.CSV_OPINIONES, low_memory=False)
    return libros, lectores, opiniones


# --------------------------------------------------------------------------------------
# Perfilado
# --------------------------------------------------------------------------------------

def porcentaje_nulos(df: pd.DataFrame) -> pd.Series:
    """% de nulos por columna, ordenado de mayor a menor."""
    return (df.isna().mean() * 100).round(2).sort_values(ascending=False)


def perfilar_tablas(
    libros: pd.DataFrame,
    lectores: pd.DataFrame,
    opiniones: pd.DataFrame,
) -> None:
    """Imprime filas, tipos, % de nulos por columna e IDs únicos de cada tabla."""
    # Para cada tabla: sus columnas de ID, y la clave que debería identificar una fila.
    tablas = [
        ("libros", libros, [config.COL_ID_LIBRO], [config.COL_ID_LIBRO]),
        ("lectores", lectores, [config.COL_ID_LECTOR], [config.COL_ID_LECTOR]),
        ("opiniones", opiniones, [config.COL_ID_LECTOR, config.COL_ID_LIBRO],
         [config.COL_ID_LECTOR, config.COL_ID_LIBRO]),
    ]

    for nombre, df, cols_id, clave in tablas:
        print(f"\n### Tabla `{nombre}` — {len(df):,} filas × {df.shape[1]} columnas")

        perfil = pd.DataFrame(
            {
                "tipo": df.dtypes.astype(str),
                "% nulos": (df.isna().mean() * 100).round(2),
                "valores únicos": df.nunique(),
            }
        )
        print(perfil.to_string())

        for col in cols_id:
            print(f"  IDs únicos en `{col}`: {df[col].nunique():,}")
        duplicados = int(df.duplicated(subset=clave).sum())
        print(f"  Filas duplicadas por la clave {clave}: {duplicados:,}"
              f"{'  <-- el merge podría multiplicar filas' if duplicados else ''}")


# --------------------------------------------------------------------------------------
# Unión
# --------------------------------------------------------------------------------------

def unir(
    libros: pd.DataFrame,
    lectores: pd.DataFrame,
    opiniones: pd.DataFrame,
    how: str = config.HOW_UNION,
) -> pd.DataFrame:
    """Une las tres tablas dejando a `opiniones` en el centro: una fila = una opinión.

    `how` decide qué pasa con una opinión cuyo libro o lector no existe en las otras
    tablas: con "left" la fila se conserva con todos los atributos en nulo, con "inner"
    se descarta. Ninguna de las dos tablas laterales tiene IDs duplicados, así que el
    merge no puede multiplicar filas.

    Las columnas `genero` de libros y de lectores colisionan: quedan como
    `genero_libro` y `genero_lector`.
    """
    return (
        opiniones
        .merge(libros, on=config.COL_ID_LIBRO, how=how)
        .merge(lectores, on=config.COL_ID_LECTOR, how=how, suffixes=("_libro", "_lector"))
    )


# --------------------------------------------------------------------------------------
# Target
# --------------------------------------------------------------------------------------

def construir_target(df: pd.DataFrame) -> pd.DataFrame:
    """Descarta los rating 6 y recién después mapea >= 7 a 1 y <= 5 a 0.

    El orden importa: el 6 es un rating gris que no aporta información, así que sale
    del dataset antes de binarizar. Se conserva la columna `rating` para trazabilidad,
    pero queda prohibida como predictora (CLAUDE.md 3.3): contiene el target.
    """
    rating = df[config.COL_RATING]
    mask = rating.notna() & rating.ne(config.RATING_DESCARTADO)

    return (
        df.loc[mask]
        .assign(**{config.TARGET: (rating[mask] >= config.RATING_MIN_GUSTO).astype("int8")})
        .reset_index(drop=True)
    )


## 3. Diagnóstico de la unión

Diagnóstico de la unión de las tres tablas. Material para el informe.

Corre la carga, el perfilado, la unión y la construcción del target, e imprime:
cobertura de libros y lectores, opiniones huérfanas, % de nulos antes y después
del merge, comparación left vs inner y distribución de clases.

Uso:  python -m src.diagnostico [--how left|inner]
      python -m src.diagnostico | tee resultados/01_diagnostico_union.txt

*(desde `src/diagnostico.py`)*


In [5]:
import pandas as pd


COLS_LIBROS = ["titulo", "autor", "genero_libro", "editorial", "anio_edicion",
               "isbn", "resumen", "img_src"]
COLS_LECTORES = ["nombre", "genero_lector", "vive_en", "nacimiento"]
COLS_OPINIONES = ["id_lector", "id_libro", "fecha", "rating"]

# Nombre en la tabla de origen -> nombre después del merge (la colisión de `genero`).
RENOMBRES = {"genero_libro": "genero", "genero_lector": "genero"}


def titulo(texto: str) -> None:
    print("\n" + "=" * 88)
    print(texto)
    print("=" * 88)


# --------------------------------------------------------------------------------------
# 1. Cobertura: cuánto de cada tabla lateral usa realmente el dataset
# --------------------------------------------------------------------------------------

def cobertura(libros: pd.DataFrame, lectores: pd.DataFrame, opiniones: pd.DataFrame) -> pd.DataFrame:
    """Cuántas entidades hay en cada tabla y cuántas aparecen en al menos una opinión."""
    filas = []
    for nombre, df, col in [("libros", libros, config.COL_ID_LIBRO),
                            ("lectores", lectores, config.COL_ID_LECTOR)]:
        en_tabla = df[col].nunique()
        referenciados = set(opiniones[col].unique())
        con_opinion = df[col].isin(referenciados).sum()
        filas.append({
            "tabla": nombre,
            "en la tabla": en_tabla,
            "con al menos una opinión": int(con_opinion),
            "% usado": round(100 * con_opinion / en_tabla, 2),
            "sin ninguna opinión": int(en_tabla - con_opinion),
            "% descartado por el merge": round(100 * (en_tabla - con_opinion) / en_tabla, 2),
        })
    return pd.DataFrame(filas).set_index("tabla")


def huerfanas(libros: pd.DataFrame, lectores: pd.DataFrame, opiniones: pd.DataFrame) -> pd.DataFrame:
    """Opiniones que apuntan a un libro o a un lector que no existe en su tabla."""
    sin_libro = ~opiniones[config.COL_ID_LIBRO].isin(set(libros[config.COL_ID_LIBRO]))
    sin_lector = ~opiniones[config.COL_ID_LECTOR].isin(set(lectores[config.COL_ID_LECTOR]))
    total = len(opiniones)

    filas = [
        ("libro inexistente", int(sin_libro.sum())),
        ("lector inexistente", int(sin_lector.sum())),
        ("libro Y lector inexistentes", int((sin_libro & sin_lector).sum())),
        ("al menos una de las dos (se pierden con inner)", int((sin_libro | sin_lector).sum())),
    ]
    return pd.DataFrame(
        [{"caso": c, "opiniones": n, "% del total": round(100 * n / total, 3)} for c, n in filas]
    ).set_index("caso")


# --------------------------------------------------------------------------------------
# 2. Nulos antes y después del merge
# --------------------------------------------------------------------------------------

def tabla_nulos(libros, lectores, opiniones, unidos: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """% de nulos por columna en la tabla de origen y en el dataset unido (left e inner).

    La columna intermedia — la tabla de origen restringida a las entidades que sí
    aparecen en opiniones — es la que separa las dos causas de nulos: los registros
    vacíos que nunca vamos a ver, y los que el merge sí arrastra al dataset.
    """
    con_opinion = {
        "libros": libros[libros[config.COL_ID_LIBRO].isin(set(opiniones[config.COL_ID_LIBRO]))],
        "lectores": lectores[lectores[config.COL_ID_LECTOR].isin(set(opiniones[config.COL_ID_LECTOR]))],
        "opiniones": opiniones,
    }
    origen = {"libros": libros, "lectores": lectores, "opiniones": opiniones}

    filas = []
    for tabla, columnas in [("opiniones", COLS_OPINIONES),
                            ("libros", COLS_LIBROS),
                            ("lectores", COLS_LECTORES)]:
        for col in columnas:
            col_origen = RENOMBRES.get(col, col)
            antes = 100 * origen[tabla][col_origen].isna().mean()
            filtrado = 100 * con_opinion[tabla][col_origen].isna().mean()
            fila = {
                "tabla": tabla,
                "columna": col,
                "% nulos ANTES (tabla completa)": round(antes, 2),
                "% nulos (sólo con opiniones)": round(filtrado, 2),
            }
            for nombre_how, df in unidos.items():
                fila[f"% nulos DESPUÉS ({nombre_how})"] = round(100 * df[col].isna().mean(), 2)
            fila["reducción (pp)"] = round(antes - 100 * unidos["left"][col].isna().mean(), 2)
            filas.append(fila)

    return pd.DataFrame(filas).set_index(["tabla", "columna"])


def resumen_nulos_libros(libros: pd.DataFrame, unidos: dict[str, pd.DataFrame]) -> None:
    """El número del informe: cuánto del 60% de nulos de `libros` nos toca limpiar."""
    cols_origen = [RENOMBRES.get(c, c) for c in COLS_LIBROS]

    celdas_tabla = libros[cols_origen].size
    nulas_tabla = int(libros[cols_origen].isna().sum().sum())

    # Un libro está "vacío" si no tiene ningún atributo cargado.
    vacios = libros[cols_origen].isna().all(axis=1)
    print(f"Celdas nulas en las columnas de `libros` (tabla completa): "
          f"{nulas_tabla:,} de {celdas_tabla:,}  ({100 * nulas_tabla / celdas_tabla:.2f} %)")
    print(f"Libros sin ningún atributo cargado (fila entera vacía): "
          f"{int(vacios.sum()):,} de {len(libros):,}  ({100 * vacios.mean():.2f} %)")

    for nombre_how, df in unidos.items():
        nulas = int(df[COLS_LIBROS].isna().sum().sum())
        print(f"Nulos en columnas de libros dentro del dataset unido ({nombre_how}): "
              f"{nulas:,} de {df[COLS_LIBROS].size:,}  ({100 * nulas / df[COLS_LIBROS].size:.2f} %)")


# --------------------------------------------------------------------------------------
# 3. left vs inner
# --------------------------------------------------------------------------------------

def comparar_how(unidos: dict[str, pd.DataFrame], opiniones: pd.DataFrame) -> pd.DataFrame:
    """Costo y beneficio de cada `how`, en filas y en calidad de las columnas."""
    filas = []
    for nombre_how, df in unidos.items():
        con_target = carga.construir_target(df)
        nulos_libros = 100 * df[COLS_LIBROS].isna().mean().mean()
        nulos_lectores = 100 * df[COLS_LECTORES].isna().mean().mean()
        filas.append({
            "how": nombre_how,
            "filas tras el merge": len(df),
            "filas tras quitar rating 6": len(con_target),
            "% de opiniones conservadas": round(100 * len(df) / len(opiniones), 3),
            "% nulos promedio (cols. libros)": round(nulos_libros, 2),
            "% nulos promedio (cols. lectores)": round(nulos_lectores, 2),
            "% clase 1 (gustó)": round(100 * con_target[config.TARGET].mean(), 2),
        })
    return pd.DataFrame(filas).set_index("how")


def target_en_huerfanas(base_left: pd.DataFrame) -> pd.DataFrame:
    """¿Las opiniones huérfanas tienen un target distinto? Si no, dropearlas no sesga."""
    df = carga.construir_target(base_left)
    es_huerfana = df["titulo"].isna() | df["nombre"].isna()
    resumen = (
        df.assign(grupo=es_huerfana.map({True: "huérfanas", False: "con libro y lector"}))
        .groupby("grupo")[config.TARGET]
        .agg(filas="size", **{"% gustó": lambda s: round(100 * s.mean(), 2)})
    )
    return resumen


# --------------------------------------------------------------------------------------
# 4. Distribución de clases
# --------------------------------------------------------------------------------------

def distribucion_clases(opiniones: pd.DataFrame, base: pd.DataFrame) -> None:
    print("Ratings en la tabla de opiniones (antes de construir el target):")
    conteo = opiniones[config.COL_RATING].value_counts().sort_index()
    tabla = pd.DataFrame({
        "opiniones": conteo,
        "% del total": (100 * conteo / len(opiniones)).round(2),
        "clase": [
            "1 (gustó)" if r >= config.RATING_MIN_GUSTO
            else "DESCARTADO" if r == config.RATING_DESCARTADO
            else "0 (no gustó)"
            for r in conteo.index
        ],
    })
    print(tabla.to_string())

    descartadas = int((opiniones[config.COL_RATING] == config.RATING_DESCARTADO).sum())
    print(f"\nFilas eliminadas por rating == 6: {descartadas:,} "
          f"({100 * descartadas / len(opiniones):.2f} % de las opiniones)")

    print(f"\nDistribución final de `{config.TARGET}` ({len(base):,} filas):")
    dist = base[config.TARGET].value_counts().sort_index()
    print(pd.DataFrame({
        "filas": dist,
        "%": (100 * dist / len(base)).round(2),
    }).to_string())
    ratio = dist.max() / dist.min()
    print(f"\nRatio de desbalanceo: {ratio:.2f} a 1 — desbalanceo clásico, no extremo.")


# --------------------------------------------------------------------------------------
# Runner
# --------------------------------------------------------------------------------------

def main_diagnostico(how: str = config.HOW_UNION) -> pd.DataFrame:
    pd.set_option("display.width", 200)
    pd.set_option("display.max_columns", 50)

    titulo("0. LECTURA DE LAS TRES TABLAS")
    print(f"libros:    {config.CSV_LIBROS}")
    print(f"lectores:  {config.CSV_LECTORES}")
    print(f"opiniones: {config.CSV_OPINIONES}")
    libros, lectores, opiniones = carga.cargar_tablas()

    titulo("1. PERFIL DE CADA TABLA POR SEPARADO")
    carga.perfilar_tablas(libros, lectores, opiniones)

    titulo("2. COBERTURA: CUÁNTO DE CADA TABLA USA EL DATASET")
    print(cobertura(libros, lectores, opiniones).to_string())
    print("\nOpiniones huérfanas (referencian un ID que no existe):")
    print(huerfanas(libros, lectores, opiniones).to_string())

    unidos = {
        "left": carga.unir(libros, lectores, opiniones, how="left"),
        "inner": carga.unir(libros, lectores, opiniones, how="inner"),
    }

    titulo("3. NULOS ANTES Y DESPUÉS DEL MERGE")
    nulos = tabla_nulos(libros, lectores, opiniones, unidos)
    print(nulos.to_string())

    titulo("4. EL 60% DE NULOS DE `libros` NO ES UN PROBLEMA A LIMPIAR")
    resumen_nulos_libros(libros, unidos)

    titulo("5. left vs inner")
    comparacion = comparar_how(unidos, opiniones)
    print(comparacion.to_string())
    print("\n¿Las opiniones huérfanas tienen otro comportamiento de target?")
    print(target_en_huerfanas(unidos["left"]).to_string())

    base = carga.construir_target(unidos[how])

    titulo(f"6. DISTRIBUCIÓN DE CLASES (dataset construido con how='{how}')")
    distribucion_clases(opiniones, base)

    titulo("7. SALIDAS")
    # Las dos tablas van al informe, así que se guardan en resultados/ (sí versionado).
    config.DIR_RESULTADOS.mkdir(parents=True, exist_ok=True)
    nulos.to_csv(config.DIR_RESULTADOS / "01_nulos_antes_despues.csv")
    comparacion.to_csv(config.DIR_RESULTADOS / "01_comparacion_left_inner.csv")
    print(f"Guardado: {config.DIR_RESULTADOS / '01_nulos_antes_despues.csv'}")
    print(f"Guardado: {config.DIR_RESULTADOS / '01_comparacion_left_inner.csv'}")

    config.DIR_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
    base.to_pickle(config.CHECKPOINT_BASE)
    print(f"Guardado: {config.CHECKPOINT_BASE}")
    print(f"Dimensiones: {base.shape[0]:,} filas × {base.shape[1]} columnas  "
          f"(how='{how}', {config.CHECKPOINT_BASE.stat().st_size / 1e6:.1f} MB)")
    print(f"Columnas: {list(base.columns)}")
    return base


## 4. Análisis exploratorio

Análisis exploratorio. No modifica los datos: sólo mira y produce figuras.

Cada figura tiene que poder acompañarse de un análisis; si no dice nada, no está.
Por eso no hay un gráfico por columna: hay un gráfico por hallazgo.

Uso:  python -m src.eda

*(desde `src/eda.py`)*


In [6]:
import matplotlib

matplotlib.use("Agg")

import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------
# Estilo
# --------------------------------------------------------------------------------------
# Paleta validada para daltonismo (ΔE CVD 9.2, visión normal 24.0 sobre las tres
# series). El orden de los slots es el mecanismo de seguridad, no decoración.

SUPERFICIE = "#fcfcfb"
TINTA = "#0b0b0b"
TINTA_SUAVE = "#52514e"
GRILLA = "#e3e2de"

AZUL = "#2a78d6"      # serie 1 — el color por defecto de todo lo univariado
NARANJA = "#eb6834"   # serie 2
AQUA = "#1baf7a"      # serie 3
ROJO = "#e34948"      # reservado: marca un problema de calidad, siempre con etiqueta
GRIS = "#9a9892"      # contexto, referencia, "resto"

plt.rcParams.update({
    "figure.facecolor": SUPERFICIE,
    "axes.facecolor": SUPERFICIE,
    "savefig.facecolor": SUPERFICIE,
    "axes.edgecolor": GRILLA,
    "axes.labelcolor": TINTA_SUAVE,
    "axes.titlecolor": TINTA,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": GRILLA,
    "grid.linewidth": 0.8,
    "xtick.color": TINTA_SUAVE,
    "ytick.color": TINTA_SUAVE,
    "text.color": TINTA,
    "font.size": 9,
    "figure.dpi": 150,
    "savefig.bbox": "tight",
})


def _limpiar(ax, ejes=("top", "right")) -> None:
    """Saca el marco: la grilla ya da la referencia, el marco sólo agrega tinta."""
    for lado in ejes:
        ax.spines[lado].set_visible(False)
    ax.grid(axis="x", visible=False)


# Contorno del color de la superficie: despega una etiqueta de la barra o de la
# línea de referencia sobre la que caiga, sin recuadro ni fondo opaco.
HALO = [pe.withStroke(linewidth=2.6, foreground=SUPERFICIE)]


def _titulo(fig, texto: str) -> None:
    """Título de la figura, reservando el espacio antes de escribirlo.

    Sin el `rect` el suptitle se superpone con el título del primer eje.
    """
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    fig.suptitle(texto, x=0.005, ha="left", fontsize=13, fontweight="bold", y=0.995)


def _guardar(fig, nombre: str) -> None:
    config.DIR_FIGURAS.mkdir(parents=True, exist_ok=True)
    destino = config.DIR_FIGURAS / f"{nombre}.png"
    fig.savefig(destino)
    plt.close(fig)
    print(f"  → {destino.name}")


def _etiquetar_barras(ax, barras, valores, fmt="{:.0%}", dx=0.0) -> None:
    """Etiqueta directa sobre cada barra: sin hover, el número tiene que estar."""
    for barra, valor in zip(barras, valores):
        ax.text(barra.get_width() + dx, barra.get_y() + barra.get_height() / 2,
                fmt.format(valor), va="center", ha="left",
                fontsize=8, color=TINTA_SUAVE, path_effects=HALO)


# --------------------------------------------------------------------------------------
# 1. Univariado: el target
# --------------------------------------------------------------------------------------

def fig_rating_y_target(opiniones: pd.DataFrame, base: pd.DataFrame) -> None:
    """El hueco del rating 6 y el desbalanceo que deja.

    Justifica las dos reglas del target: por qué se descarta el 6 y por qué la
    métrica no puede ser accuracy.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8), width_ratios=[2, 1])

    conteo = opiniones[config.COL_RATING].value_counts().sort_index()
    colores = [ROJO if r == config.RATING_DESCARTADO else AZUL for r in conteo.index]
    ax1.bar(conteo.index, conteo.values, color=colores, width=0.7)
    ax1.set_title("Distribución del rating, antes de construir el target")
    ax1.set_xlabel("rating"); ax1.set_ylabel("opiniones")
    ax1.set_xticks(range(1, 11))
    ax1.yaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}k")
    descartadas = int(conteo.get(config.RATING_DESCARTADO, 0))
    ax1.annotate(f"rating 6 — se descarta\n{descartadas:,} opiniones ({descartadas/len(opiniones):.1%})",
                 xy=(6, descartadas), xytext=(3.2, descartadas * 1.06),
                 color=ROJO, fontsize=8.5, fontweight="bold",
                 arrowprops=dict(arrowstyle="-", color=ROJO, lw=1.2))
    _limpiar(ax1)

    dist = base[config.TARGET].value_counts().sort_index()
    barras = ax2.bar(["0 · no gustó", "1 · gustó"], dist.values, color=[NARANJA, AZUL], width=0.6)
    ax2.set_title("Target `gusto`")
    ax2.set_ylabel("opiniones")
    ax2.yaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}k")
    for barra, valor in zip(barras, dist.values):
        ax2.text(barra.get_x() + barra.get_width() / 2, valor, f"{valor/len(base):.1%}",
                 ha="center", va="bottom", fontsize=9, fontweight="bold", color=TINTA)
    ax2.set_ylim(0, dist.max() * 1.15)
    _limpiar(ax2)

    _titulo(fig, "El rating gris se elimina y deja un problema desbalanceado 84/16")
    _guardar(fig, "01_rating_y_target")


# --------------------------------------------------------------------------------------
# 2. Univariado: las numéricas, y sus valores imposibles
# --------------------------------------------------------------------------------------

def fig_nacimiento_y_edad(base: pd.DataFrame) -> None:
    """El pico de 1910 es un centinela del formulario, no gente de 116 años."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

    lectores = base.drop_duplicates("id_lector")
    conteo = lectores.nacimiento.value_counts().sort_index()
    colores = [ROJO if a == 1910 else AZUL for a in conteo.index]
    ax1.bar(conteo.index, conteo.values, color=colores, width=1.0)
    ax1.set_title("Año de nacimiento declarado (lectores únicos)")
    ax1.set_xlabel("año de nacimiento"); ax1.set_ylabel("lectores")
    pico = int(conteo.get(1910, 0))
    ax1.annotate(f"1910: {pico} lectores\nes el mínimo del selector del sitio,\nno una cohorte real",
                 xy=(1910, pico), xytext=(1925, pico * 0.85),
                 color=ROJO, fontsize=8.5, fontweight="bold",
                 arrowprops=dict(arrowstyle="-", color=ROJO, lw=1.2))
    _limpiar(ax1)

    edad = pd.to_datetime(base.fecha, format="%d-%m-%Y").dt.year - base.nacimiento
    ax2.hist(edad.dropna(), bins=np.arange(-10, 125, 2), color=AZUL)
    ax2.set_title("Edad del lector al momento de opinar")
    ax2.set_xlabel("edad (años)"); ax2.set_ylabel("opiniones")
    ax2.yaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}k")
    for limite, etiqueta in [(10, "menores de 10"), (90, "90 o más")]:
        ax2.axvline(limite, color=ROJO, lw=1.2, ls="--")
    n_bajo = int((edad < 10).sum()); n_alto = int((edad >= 90).sum())
    ax2.annotate(f"{n_bajo:,} opiniones con edad < 10\n"
                 f"{n_alto:,} con edad ≥ 90 — son la joroba aislada\n"
                 f"de la derecha, casi toda del centinela 1910",
                 xy=(0.98, 0.97), xycoords="axes fraction", ha="right", va="top",
                 color=ROJO, fontsize=8.5, fontweight="bold", path_effects=HALO)
    _limpiar(ax2)

    _titulo(fig, "`nacimiento` tiene un valor centinela que contamina toda la edad")
    _guardar(fig, "02_nacimiento_y_edad")


def fig_anio_edicion(base: pd.DataFrame) -> None:
    """La edición que guarda el sitio es la última, no la que leyó el lector."""
    anio = pd.to_numeric(base.anio_edicion, errors="coerce")
    anio_opinion = pd.to_datetime(base.fecha, format="%d-%m-%Y").dt.year

    fig, ax = plt.subplots(figsize=(9, 3.8))
    validos = anio[(anio >= 1900) & (anio <= 2026)]
    ax.hist(validos, bins=np.arange(1900, 2028, 1), color=AZUL)
    ax.set_title("Año de edición del libro")
    ax.set_xlabel("año de edición"); ax.set_ylabel("opiniones")
    ax.yaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}k")

    posteriores = int((anio > anio_opinion).sum())
    fuera = int(((anio < 1900) | (anio > 2026)).sum() + (anio.isna() & base.anio_edicion.notna()).sum())
    ax.annotate(
        f"{posteriores:,} opiniones ({posteriores/len(base):.1%}) tienen un año de edición\n"
        f"POSTERIOR a la fecha de la opinión: el sitio guarda la última edición\n"
        f"del catálogo, no el ejemplar que se leyó.\n"
        f"Otras {fuera:,} tienen el año ilegible o fuera de rango.",
        xy=(0.02, 0.95), xycoords="axes fraction", va="top",
        color=ROJO, fontsize=8.5, fontweight="bold")
    _limpiar(ax)
    _titulo(fig, "`anio_edicion` no es el año en que se leyó el libro")
    _guardar(fig, "03_anio_edicion")


# --------------------------------------------------------------------------------------
# 3. Univariado: frecuencias categóricas
# --------------------------------------------------------------------------------------

def fig_frecuencias_categoricas(base: pd.DataFrame) -> None:
    """Qué domina cada categórica. Muestra la concentración y la cola sucia."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    paneles = [
        (axes[0][0], "genero_libro", "Género literario (top 12)", 12),
        (axes[0][1], "genero_lector", "Género del lector — tal como viene", None),
        (axes[1][0], "editorial", "Editorial (top 12)", 12),
        (axes[1][1], "autor", "Autor (top 12)", 12),
    ]
    for ax, col, titulo, top in paneles:
        conteo = base[col].value_counts(dropna=False)
        if top:
            conteo = conteo.head(top)
        etiquetas = [("(nulo)" if pd.isna(i) else str(i))[:38] for i in conteo.index]
        colores = [ROJO if (pd.isna(i) or str(i).strip() in {"-", "¿?"}) else AZUL
                   for i in conteo.index]
        barras = ax.barh(etiquetas[::-1], conteo.values[::-1], color=colores[::-1], height=0.72)
        ax.set_title(titulo)
        ax.xaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}k")
        _etiquetar_barras(ax, barras, conteo.values[::-1] / len(base),
                          dx=conteo.max() * 0.01)
        ax.set_xlim(0, conteo.max() * 1.18)
        ax.tick_params(labelsize=8)
        _limpiar(ax)

    axes[0][1].annotate('el literal "-" es un nulo disfrazado',
                        xy=(0.35, 0.15), xycoords="axes fraction",
                        color=ROJO, fontsize=8.5, fontweight="bold")
    _titulo(fig, "Frecuencias por categoría — en rojo, los nulos disfrazados")
    _guardar(fig, "04_frecuencias_categoricas")


# --------------------------------------------------------------------------------------
# 4. Cardinalidad: lo que condiciona todo lo que venga después
# --------------------------------------------------------------------------------------

def fig_cardinalidad(base: pd.DataFrame) -> pd.DataFrame:
    """Cuánta cobertura da el top-N de cada categórica.

    Es la figura que decide la estrategia de dummies: con 19.500 autores no se
    puede hacer one-hot, pero si el top 100 cubre buena parte de las opiniones,
    un top-N + "otros" sí es viable.
    """
    columnas = [("autor", AZUL), ("editorial", NARANJA), ("genero_libro", AQUA)]
    fig, ax = plt.subplots(figsize=(9.5, 4.6))

    filas = []
    for col, color in columnas:
        conteo = base[col].value_counts()
        acumulado = conteo.cumsum() / len(base)
        x = np.arange(1, len(acumulado) + 1)
        ax.plot(x, acumulado.values, color=color, lw=2, solid_capstyle="round")
        # Etiqueta directa al final de la curva: el aqua no llega a 3:1 de
        # contraste, así que la identidad no puede depender sólo del color.
        ax.annotate(f"{col} ({conteo.size:,})",
                    xy=(len(acumulado), acumulado.iloc[-1]),
                    xytext=(6, 0), textcoords="offset points",
                    color=color, fontsize=9, fontweight="bold", va="center")
        fila = {"columna": col, "categorías": conteo.size}
        for n in (10, 50, 100):
            fila[f"top {n}"] = round(float(acumulado.iloc[min(n, len(acumulado)) - 1]), 4)
        filas.append(fila)

    for n in (10, 50, 100):
        ax.axvline(n, color=GRIS, lw=0.9, ls=":")
        ax.text(n, 1.02, f"top {n}", ha="center", fontsize=8, color=TINTA_SUAVE)

    ax.set_xscale("log")
    ax.set_xlim(1, 3e4)
    ax.set_ylim(0, 1.06)
    ax.set_xlabel("cantidad de categorías incluidas (escala logarítmica)")
    ax.set_ylabel("cobertura de las opiniones")
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
    ax.set_title("Cobertura acumulada del top-N de cada categórica")
    _limpiar(ax)
    _titulo(fig, "La cardinalidad decide la estrategia de dummies")
    _guardar(fig, "05_cardinalidad_acumulada")
    return pd.DataFrame(filas).set_index("columna")


# --------------------------------------------------------------------------------------
# 5. Actividad: la cola larga
# --------------------------------------------------------------------------------------

def fig_actividad(base: pd.DataFrame) -> pd.DataFrame:
    """Opiniones por lector y por libro. La cola es el hallazgo, no el promedio."""
    por_lector = base.id_lector.value_counts()
    por_libro = base.id_libro.value_counts()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
    for ax, serie, etiqueta, color in [(ax1, por_lector, "lector", AZUL),
                                       (ax2, por_libro, "libro", NARANJA)]:
        bins = np.logspace(0, np.log10(serie.max()) + 0.05, 45)
        ax.hist(serie.values, bins=bins, color=color)
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_title(f"Opiniones por {etiqueta}")
        ax.set_xlabel(f"opiniones del {etiqueta} (log)")
        ax.set_ylabel(f"cantidad de {etiqueta}es (log)" if etiqueta == "lector"
                      else "cantidad de libros (log)")
        ax.annotate(
            f"mediana {serie.median():.0f} · máximo {serie.max():,}\n"
            f"{(serie == 1).mean():.0%} tiene una sola opinión",
            xy=(0.97, 0.95), xycoords="axes fraction", ha="right", va="top",
            fontsize=8.5, color=TINTA_SUAVE, path_effects=HALO)
        _limpiar(ax)
        ax.grid(axis="x", visible=True)

    _titulo(fig, "La actividad es una cola larga: pocos lectores explican muchas opiniones")
    _guardar(fig, "06_actividad_lector_y_libro")

    resumen = pd.DataFrame({
        "opiniones por lector": por_lector.describe(percentiles=[.5, .9, .99]),
        "opiniones por libro": por_libro.describe(percentiles=[.5, .9, .99]),
    }).round(1)
    return resumen


# --------------------------------------------------------------------------------------
# 6. Bivariado contra el target
# --------------------------------------------------------------------------------------

def _tasa_por(base: pd.DataFrame, col: pd.Series, minimo: int = 500) -> pd.DataFrame:
    """Tasa de `gusto` por categoría, descartando las categorías sin volumen.

    Una categoría con 12 opiniones puede dar 100% de gusto por azar; incluirla
    ensucia el gráfico y sugiere una relación que no existe.
    """
    tabla = (base.assign(_cat=col)
             .groupby("_cat", dropna=False)[config.TARGET]
             .agg(tasa="mean", n="size"))
    return tabla[tabla.n >= minimo].sort_values("tasa")


def fig_tasa_por_genero_literario(base: pd.DataFrame) -> pd.DataFrame:
    """El género del libro sí mueve la aguja: hay 20 puntos entre extremos."""
    tabla = _tasa_por(base, base.genero_libro, minimo=1000)
    global_ = base[config.TARGET].mean()

    fig, ax = plt.subplots(figsize=(9, 6))
    colores = [AZUL if t >= global_ else NARANJA for t in tabla.tasa]
    barras = ax.barh([str(i)[:40] for i in tabla.index], tabla.tasa, color=colores, height=0.72)
    ax.axvline(global_, color=TINTA_SUAVE, lw=1.4, ls="--")
    ax.text(global_, len(tabla) - 0.2, f"  media global {global_:.1%}",
            fontsize=8.5, color=TINTA_SUAVE, va="top", path_effects=HALO)
    _etiquetar_barras(ax, barras, tabla.tasa.values, dx=0.004)
    for barra, n in zip(barras, tabla.n.values):
        ax.text(0.006, barra.get_y() + barra.get_height() / 2, f"n={n:,}",
                va="center", fontsize=7.5, color=SUPERFICIE)
    ax.set_xlim(0, 1.0)
    ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
    ax.set_xlabel("tasa de «le gustó»")
    ax.set_title("Tasa de «le gustó» por género literario (categorías con n ≥ 1.000)")
    ax.tick_params(labelsize=8.5)
    _limpiar(ax)
    rango = (tabla.tasa.max() - tabla.tasa.min()) * 100
    _titulo(fig, f"El género del libro discrimina: {rango:.0f} puntos entre el mejor "
                 f"({tabla.index[-1]}) y el peor ({tabla.index[0]})")
    _guardar(fig, "07_tasa_gusto_por_genero_literario")
    return tabla


def fig_tasa_por_decada(base: pd.DataFrame) -> pd.DataFrame:
    """La década de edición contra el target."""
    anio = pd.to_numeric(base.anio_edicion, errors="coerce")
    anio = anio.where((anio >= 1900) & (anio <= 2026))
    decada = (anio // 10 * 10)
    tabla = _tasa_por(base, decada, minimo=500)
    tabla = tabla.sort_index()

    fig, ax = plt.subplots(figsize=(9, 3.8))
    ax.plot(tabla.index, tabla.tasa, color=AZUL, lw=2, marker="o", ms=6,
            mfc=AZUL, mec=SUPERFICIE, mew=1.5)
    ax.axhline(base[config.TARGET].mean(), color=TINTA_SUAVE, lw=1.2, ls="--")
    ax.text(tabla.index.max(), base[config.TARGET].mean(),
            f"media global {base[config.TARGET].mean():.1%} ", fontsize=8.5,
            color=TINTA_SUAVE, va="top", ha="right", path_effects=HALO)
    for x, y, n in zip(tabla.index, tabla.tasa, tabla.n):
        ax.annotate(f"{y:.1%}", (x, y), textcoords="offset points", xytext=(0, 8),
                    ha="center", fontsize=8, color=TINTA_SUAVE, path_effects=HALO)
    # La escala NO se ajusta al rango de los datos: con 1,9 puntos de diferencia, un
    # eje pegado a los datos dibujaría una montaña y sugeriría un efecto que no está.
    # El eje fijo muestra lo que hay, que es una recta.
    ax.set_ylim(0.6, 1.0)
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
    ax.set_xlabel("década de edición"); ax.set_ylabel("tasa de «le gustó»")
    ax.set_title("Tasa de «le gustó» por década de edición (n ≥ 500)")
    _limpiar(ax)
    ax.grid(axis="x", visible=True)
    rango = (tabla.tasa.max() - tabla.tasa.min()) * 100
    _titulo(fig, f"La década de edición casi no mueve la aguja: {rango:.1f} puntos "
                 f"entre extremos")
    _guardar(fig, "08_tasa_gusto_por_decada")
    return tabla


def fig_tasa_por_perfil_lector(base: pd.DataFrame) -> pd.DataFrame:
    """Género del lector y actividad del lector contra el target."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8), width_ratios=[1, 1.4])
    global_ = base[config.TARGET].mean()

    genero = _tasa_por(base, base.genero_lector, minimo=500)
    colores = [ROJO if str(i).strip() == "-" else AZUL for i in genero.index]
    barras = ax1.barh([("(nulo)" if pd.isna(i) else str(i)) for i in genero.index],
                      genero.tasa, color=colores, height=0.6)
    ax1.axvline(global_, color=TINTA_SUAVE, lw=1.4, ls="--")
    _etiquetar_barras(ax1, barras, genero.tasa.values, dx=0.004)
    ax1.set_xlim(0, 1.0)
    ax1.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
    ax1.set_title("Por género del lector")
    ax1.set_xlabel("tasa de «le gustó»")
    _limpiar(ax1)

    por_lector = base.id_lector.value_counts()
    cortes = [0, 1, 5, 20, 100, 500, np.inf]
    etiquetas = ["1", "2–5", "6–20", "21–100", "101–500", "500+"]
    tramo = pd.cut(base.id_lector.map(por_lector), bins=cortes, labels=etiquetas)
    actividad = _tasa_por(base, tramo, minimo=500).reindex(etiquetas).dropna()

    ax2.plot(range(len(actividad)), actividad.tasa, color=NARANJA, lw=2,
             marker="o", ms=7, mfc=NARANJA, mec=SUPERFICIE, mew=1.5)
    ax2.axhline(global_, color=TINTA_SUAVE, lw=1.2, ls="--")
    ax2.set_xticks(range(len(actividad)))
    ax2.set_xticklabels(actividad.index)
    for i, (y, n) in enumerate(zip(actividad.tasa, actividad.n)):
        ax2.annotate(f"{y:.0%}\nn={n/1000:.0f}k", (i, y), textcoords="offset points",
                     xytext=(0, 9), ha="center", fontsize=8, color=TINTA_SUAVE,
                     path_effects=HALO)
    ax2.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
    ax2.set_xlabel("opiniones totales del lector")
    ax2.set_title("Por actividad del lector")
    ax2.set_ylim(actividad.tasa.min() - 0.06, actividad.tasa.max() + 0.09)
    _limpiar(ax2)
    ax2.grid(axis="x", visible=True)

    _titulo(fig, "El lector que más opina es más exigente: la tasa cae con la actividad")
    _guardar(fig, "09_tasa_gusto_por_perfil_lector")
    return actividad


# --------------------------------------------------------------------------------------
# 7. Calidad de datos, con evidencia
# --------------------------------------------------------------------------------------

def fig_calidad(base: pd.DataFrame) -> pd.DataFrame:
    """Inventario de problemas detectados, medido en opiniones afectadas."""
    anio = pd.to_numeric(base.anio_edicion, errors="coerce")
    anio_opinion = pd.to_datetime(base.fecha, format="%d-%m-%Y").dt.year
    edad = anio_opinion - base.nacimiento
    nombres_raros = base.nombre.fillna("").str.contains(r"\d", regex=True)

    problemas = [
        ('`genero_lector` con el literal "-"', int((base.genero_lector == "-").sum())),
        ("`nacimiento` nulo", int(base.nacimiento.isna().sum())),
        ("`nacimiento` = 1910 (centinela)", int((base.nacimiento == 1910).sum())),
        ("edad al opinar fuera de [10, 90]", int(((edad < 10) | (edad > 90)).sum())),
        ("`anio_edicion` posterior a la opinión", int((anio > anio_opinion).sum())),
        ("`vive_en` nulo", int(base.vive_en.isna().sum())),
        ('`vive_en` con el literal "¿?"', int((base.vive_en == "¿?").sum())),
        ("`vive_en` sin separador ciudad-país", int((base.vive_en.notna() & ~base.vive_en.str.contains(" - ", na=False)).sum())),
        ("`nombre` con dígitos (no es un nombre)", int(nombres_raros.sum())),
        ("libro inexistente (título nulo)", int(base.titulo.isna().sum())),
        ("`resumen` nulo", int(base.resumen.isna().sum())),
    ]
    tabla = (pd.DataFrame(problemas, columns=["problema", "opiniones"])
             .assign(**{"% del dataset": lambda t: (t.opiniones / len(base) * 100).round(2)})
             .sort_values("opiniones"))

    fig, ax = plt.subplots(figsize=(10, 5))
    barras = ax.barh(tabla.problema, tabla.opiniones, color=ROJO, height=0.7)
    _etiquetar_barras(ax, barras, tabla.opiniones / len(base), dx=len(base) * 0.004)
    ax.set_xlim(0, tabla.opiniones.max() * 1.2)
    ax.xaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}k")
    ax.set_xlabel("opiniones afectadas")
    ax.set_title("Problemas de calidad detectados, medidos en opiniones afectadas")
    ax.tick_params(labelsize=8.5)
    _limpiar(ax)
    _titulo(fig, "Qué hay para limpiar, y cuánto pesa cada cosa")
    _guardar(fig, "10_calidad_de_datos")
    return tabla.sort_values("opiniones", ascending=False).reset_index(drop=True)


def evidencia_calidad(base: pd.DataFrame) -> None:
    """Los ejemplos concretos que respaldan el gráfico anterior."""
    lectores = base.drop_duplicates("id_lector")

    print("\n— Nombres que no son nombres (rompen la imputación de género por nombre):")
    raros = lectores.nombre.dropna()
    raros = raros[raros.str.contains(r"\d", regex=True)]
    print(f"  {len(raros)} de {lectores.nombre.notna().sum()} lectores. Ejemplos: {raros.head(12).tolist()}")

    print("\n— `vive_en`: formatos mezclados en la misma columna:")
    v = base.vive_en.dropna()
    print(f"  con 'ciudad - país': {v.str.contains(' - ').mean():.1%}")
    ejemplos = ["Madrid - España", "España", "¿?", "Tenerife -", "France, French Republic"]
    for e in ejemplos:
        print(f"    {e!r:35} → {int((base.vive_en == e).sum()):,} opiniones")

    print("\n— Registros que no son libros:")
    libros = base.drop_duplicates("id_libro")
    sin_titulo = libros.titulo.isna().sum()
    print(f"  {sin_titulo} id_libro con opiniones no existen en la tabla de libros")
    print("  (no se encontró un volumen relevante de revistas/agendas mal catalogadas:")
    print("   el problema no es qué tipo de registro es, es que el registro no está)")

    print("\n— Género literario: la misma categoría escrita de varias formas:")
    g = base.genero_libro.dropna()
    print(f"  {g.nunique()} categorías → {g.str.lower().str.strip().nunique()} tras normalizar mayúsculas")
    conteo = g.value_counts()
    for cat in ["No Ficción", "No ficción", "Lecturas complementarias", "Lecturas Complementarias"]:
        if cat in conteo:
            print(f"    {cat!r:32} → {conteo[cat]:,}")


# --------------------------------------------------------------------------------------
# Runner
# --------------------------------------------------------------------------------------

def main_eda() -> None:
    pd.set_option("display.width", 200)
    print("Cargando el dataset base y la tabla de opiniones cruda…")
    base = pd.read_pickle(config.CHECKPOINT_BASE)
    # La tabla cruda hace falta sólo para mostrar el hueco del rating 6, que en el
    # checkpoint ya no está.
    _, _, opiniones = carga.cargar_tablas()
    print(f"base: {len(base):,} filas · opiniones crudas: {len(opiniones):,}\n")

    print("Figuras:")
    fig_rating_y_target(opiniones, base)
    fig_nacimiento_y_edad(base)
    fig_anio_edicion(base)
    fig_frecuencias_categoricas(base)
    cardinalidad = fig_cardinalidad(base)
    actividad = fig_actividad(base)
    genero = fig_tasa_por_genero_literario(base)
    decada = fig_tasa_por_decada(base)
    perfil = fig_tasa_por_perfil_lector(base)
    calidad = fig_calidad(base)

    print("\n" + "=" * 88)
    print("CARDINALIDAD — cobertura de las opiniones por el top-N")
    print("=" * 88)
    print(cardinalidad.to_string())

    print("\n" + "=" * 88)
    print("ACTIVIDAD")
    print("=" * 88)
    print(actividad.to_string())

    print("\n" + "=" * 88)
    print("TASA DE «LE GUSTÓ» POR GÉNERO LITERARIO")
    print("=" * 88)
    print(genero.assign(tasa=lambda t: (t.tasa * 100).round(1)).to_string())

    print("\n" + "=" * 88)
    print("TASA POR DÉCADA Y POR ACTIVIDAD DEL LECTOR")
    print("=" * 88)
    print(decada.assign(tasa=lambda t: (t.tasa * 100).round(1)).to_string())
    print()
    print(perfil.assign(tasa=lambda t: (t.tasa * 100).round(1)).to_string())

    print("\n" + "=" * 88)
    print("CALIDAD DE DATOS")
    print("=" * 88)
    print(calidad.to_string(index=False))
    evidencia_calidad(base)

    config.DIR_RESULTADOS.mkdir(parents=True, exist_ok=True)
    calidad.to_csv(config.DIR_RESULTADOS / "03_problemas_de_calidad.csv", index=False)
    cardinalidad.to_csv(config.DIR_RESULTADOS / "03_cardinalidad.csv")
    print(f"\nTablas guardadas en {config.DIR_RESULTADOS}")


## 5. Diccionario de género por nombre

Mapa nombre de pila → género, generado una sola vez fuera del entregable.

CLAUDE.md 3.1 prohíbe `gender-guesser` y cualquier librería que no se haya visto en
clase, y propone el patrón de ejecutar la herramienta una vez afuera y pegar el
resultado como diccionario. Acá la "herramienta" no es una librería externa: es el
propio dataset. El mapa sale de los 7,056 lectores que SÍ declararon su género,
agrupando por nombre de pila.

Criterio de inclusión: el nombre aparece en al menos 2 lectores y al menos el 90% de
ellos declara el mismo género. Con umbral de 1 lector el diccionario llegaría a 2.916
entradas y 78,6% de cobertura, pero cada entrada se apoyaría en una sola persona, que
no es consenso sino anécdota.

Resultado: 596 nombres (258 M, 338 F) que cubren el 71,1% de los lectores sin
género declarado. El resto queda en "Desconocido", que es una categoría legítima.

Este archivo es DATOS, no lógica: no se edita a mano, se regenera.

*(desde `src/genero_por_nombre.py`)*


In [7]:
GENERO_POR_NOMBRE = {
    "aarón": "M", "abel": "M", "abraham": "M", "ada": "F", "adolfo": "M", "adrian": "M",
    "adriana": "F", "adrià": "M", "adrián": "M", "agostina": "F", "agus": "M", "agustin": "M",
    "aida": "F", "aina": "F", "ainhoa": "F", "ainoa": "F", "aitor": "M", "alba": "F", "albert":
    "M", "alberto": "M", "aldo": "M", "alejandra": "F", "alejandro": "M", "alejo": "M",
    "alessandra": "F", "alex": "M", "alexander": "M", "alexandra": "F", "alexis": "M",
    "alfonso": "M", "alfredo": "M", "alicia": "F", "alma": "F", "almudena": "F", "alonso": "M",
    "alvaro": "M", "amaia": "F", "amalia": "F", "amelia": "F", "amparo": "F", "ana": "F",
    "anabel": "F", "anastasia": "F", "andrea": "F", "andres": "M", "andrés": "M", "angel": "M",
    "angela": "F", "angeles": "F", "angie": "F", "angélica": "F", "anibal": "M", "anita": "F",
    "anna": "F", "annie": "F", "anto": "F", "antonia": "F", "antonio": "M", "anuska": "F",
    "arantxa": "F", "ari": "F", "ariel": "M", "armando": "M", "arnau": "M", "arsenio": "M",
    "arturo": "M", "asun": "F", "aureliano": "M", "aurelio": "M", "axel": "M", "ayelen": "F",
    "ayelén": "F", "bastian": "M", "bea": "F", "beatriz": "F", "bego": "F", "belen": "F",
    "belén": "F", "ben": "M", "bernardo": "M", "billy": "M", "blanca": "F", "borja": "M",
    "brenda": "F", "brian": "M", "bruno": "M", "bryan": "M", "bárbara": "F", "cami": "F",
    "camila": "F", "camilo": "M", "cande": "F", "carina": "F", "carla": "F", "carles": "M",
    "carlo": "M", "carlos": "M", "carlota": "F", "carmelo": "M", "carmen": "F", "carol": "F",
    "carolina": "F", "cat": "F", "cata": "F", "catalina": "F", "cecilia": "F", "celeste": "F",
    "celia": "F", "cesar": "M", "charlotte": "F", "chema": "M", "chica": "F", "chorche": "M",
    "christian": "M", "clara": "F", "claudia": "F", "concha": "F", "conchi": "F", "conor": "M",
    "constanza": "F", "coral": "F", "cristian": "M", "cristina": "F", "curro": "M", "cynthia":
    "F", "césar": "M", "damaris": "F", "dana": "F", "daniel": "M", "daniela": "F", "david": "M",
    "delfina": "F", "delia": "F", "desiree": "F", "diana": "F", "diego": "M", "dolores": "F",
    "douglas": "M", "dulce": "F", "désirée": "F", "edgar": "M", "edison": "M", "edith": "F",
    "eduardo": "M", "edwin": "M", "el": "M", "elena": "F", "eli": "F", "elisa": "F", "elisabet":
    "F", "elisabeth": "F", "eliseo": "M", "elizabeth": "F", "elvira": "F", "elías": "M",
    "emanuel": "M", "emilia": "F", "emilio": "M", "emma": "F", "encarni": "F", "enrique": "M",
    "eric": "M", "erick": "M", "erika": "F", "ernesto": "M", "erwin": "M", "esperanza": "F",
    "esteban": "M", "estefanía": "F", "estela": "F", "ester": "F", "esther": "F", "estrella":
    "F", "euge": "F", "eugenia": "F", "eugenio": "M", "eva": "F", "ezequiel": "M", "fabian":
    "M", "fabio": "M", "fabiola": "F", "fco.": "M", "fede": "M", "federico": "M", "felipe": "M",
    "felix": "M", "fermin": "M", "fernanda": "F", "fernando": "M", "fidel": "M", "fiorella":
    "F", "flavia": "F", "flor": "F", "florencia": "F", "fly": "M", "fran": "M", "francesco":
    "M", "francis": "M", "francisco": "M", "franco": "M", "frank": "M", "fátima": "F", "félix":
    "M", "gabriel": "M", "gabriela": "F", "gem": "F", "gema": "F", "gemma": "F", "genesis": "F",
    "gerard": "M", "gerardo": "M", "german": "M", "germán": "M", "gisela": "F", "glauka": "F",
    "gloria": "F", "gonzalo": "M", "gorka": "M", "graciela": "F", "greta": "F", "guadalupe":
    "F", "guillem": "M", "guillermo": "M", "gus": "M", "gustavo": "M", "hector": "M", "helena":
    "F", "henry": "M", "hernan": "M", "hilda": "F", "hiram": "M", "hugo": "M", "héctor": "M",
    "ignacio": "M", "igor": "M", "iker": "M", "ines": "F", "ingrid": "F", "inma": "F",
    "inmaculada": "F", "inés": "F", "irene": "F", "iria": "F", "irina": "F", "irving": "M",
    "isa": "F", "isaac": "M", "isabel": "F", "ismael": "M", "israel": "M", "itziar": "F",
    "iulmi": "F", "ivan": "M", "iván": "M", "iñaki": "M", "iñigo": "M", "jackeline": "F",
    "jacobo": "M", "jacqueline": "F", "jaime": "M", "janire": "F", "jannet": "F", "jaume": "M",
    "javi": "M", "javier": "M", "jean": "M", "jeisson": "M", "jenifer": "F", "jennifer": "F",
    "jessica": "F", "jesus": "M", "jesús": "M", "jhon": "M", "jim": "M", "joaquin": "M",
    "joaquín": "M", "joel": "M", "johan": "M", "johana": "F", "johanna": "F", "john": "M",
    "jonathan": "M", "jordi": "M", "jorge": "M", "jose": "M", "joseba": "M", "joseca": "M",
    "josefa": "F", "josep": "M", "josh": "M", "josué": "M", "josé": "M", "jota": "M", "jp": "M",
    "juan": "M", "juana": "F", "juanjo": "M", "judit": "F", "judith": "F", "julia": "F",
    "julian": "M", "julieta": "F", "julio": "M", "june": "F", "karen": "F", "karin": "F",
    "karina": "F", "karla": "F", "karlos": "M", "karol": "F", "katherine": "F", "keef": "M",
    "kevin": "M", "kike": "M", "la": "F", "lady": "F", "laia": "F", "lara": "F", "lau": "F",
    "laura": "F", "lautaro": "M", "lectora": "F", "leia": "F", "leonardo": "M", "leslie": "F",
    "leti": "F", "leticia": "F", "libros": "M", "lidia": "F", "lili": "F", "lilian": "F",
    "liliana": "F", "lily": "F", "lina": "F", "linda": "F", "lizeth": "F", "lola": "F", "loles":
    "F", "loli": "F", "loly": "F", "lore": "F", "lorena": "F", "lorenzo": "M", "lourdes": "F",
    "lucas": "M", "lucia": "F", "luciana": "F", "luciano": "M", "lucila": "F", "lucy": "F",
    "lucía": "F", "ludmila": "F", "luis": "M", "luisa": "F", "luna": "F", "luz": "F",
    "macarena": "F", "maialen": "F", "maika": "F", "maite": "F", "manoli": "F", "manolo": "M",
    "manu": "M", "manuel": "M", "manuela": "F", "mar": "F", "mara": "F", "marc": "M", "marcela":
    "F", "marcelo": "M", "marco": "M", "marcos": "M", "marga": "F", "margarita": "F", "mari":
    "F", "maria": "F", "mariajo": "F", "marian": "F", "mariana": "F", "mariano": "M", "maribel":
    "F", "maricela": "F", "mariel": "F", "mariela": "F", "marimar": "F", "marina": "F", "mario":
    "M", "marisa": "F", "marisol": "F", "mark": "M", "marta": "F", "martin": "M", "martina":
    "F", "martín": "M", "maru": "F", "mary": "F", "maría": "F", "massiel": "F", "mati": "F",
    "matias": "M", "matilde": "F", "matías": "M", "mauricio": "M", "mauro": "M", "maximiliano":
    "M", "mayra": "F", "mayte": "F", "mei": "F", "melanie": "F", "melina": "F", "melisa": "F",
    "mercedes": "F", "merche": "F", "meri": "F", "meritxell": "F", "mi": "F", "micaela": "F",
    "michael": "M", "michelle": "F", "miguel": "M", "mike": "M", "mila": "F", "milena": "F",
    "mimi": "F", "mina": "F", "mireia": "F", "miren": "F", "miriam": "F", "mirian": "F", "mj":
    "F", "moises": "M", "moisés": "M", "monica": "F", "montse": "F", "montserrat": "F",
    "morgana": "F", "mr.": "M", "mª": "F", "mónica": "F", "nacho": "M", "nadia": "F", "naiara":
    "F", "nando": "M", "nat": "F", "natalia": "F", "nataly": "F", "nati": "F", "natividad": "F",
    "naty": "F", "nazaret": "F", "nerea": "F", "nestor": "M", "nicol": "F", "nicolas": "M",
    "nicole": "F", "nicolás": "M", "nieves": "F", "noelia": "F", "noemi": "F", "noemí": "F",
    "nora": "F", "nuria": "F", "néstor": "M", "núria": "F", "olga": "F", "oliver": "M", "omar":
    "M", "orlando": "M", "oscar": "M", "osvaldo": "M", "pablo": "M", "paco": "M", "paloma": "F",
    "pam": "F", "pamela": "F", "pao": "F", "paola": "F", "paqui": "F", "patri": "F", "patricia":
    "F", "paul": "M", "paula": "F", "pedro": "M", "pepe": "M", "pia": "F", "pilar": "F", "pol":
    "M", "rachel": "F", "rafa": "M", "rafael": "M", "ramiro": "M", "ramon": "M", "ramón": "M",
    "raquel": "F", "raul": "M", "raúl": "M", "rebeca": "F", "rebecca": "F", "reyes": "F",
    "ricardo": "M", "richard": "M", "ritchie": "M", "ro": "F", "rober": "M", "roberto": "M",
    "robinson": "M", "rocio": "F", "rocío": "F", "rodolfo": "M", "rodrigo": "M", "romina": "F",
    "ronald": "M", "rosa": "F", "rosana": "F", "rosario": "F", "rose": "F", "rosita": "F",
    "rousely": "F", "roxana": "F", "ruben": "M", "rubén": "M", "runa": "F", "ruth": "F",
    "salva": "M", "salvador": "M", "sam": "M", "samuel": "M", "sandra": "F", "sandro": "M",
    "sandy": "F", "santi": "M", "santiago": "M", "sara": "F", "sarah": "F", "saúl": "M",
    "sebastian": "M", "sebastián": "M", "sergi": "M", "sergio": "M", "señor": "M", "sharon":
    "F", "sheila": "F", "silvia": "F", "simón": "M", "sine": "F", "sir": "M", "sofía": "F",
    "sol": "F", "soledad": "F", "sonia": "F", "sophie": "F", "steven": "M", "susana": "F",
    "susi": "F", "tamara": "F", "tania": "F", "tere": "F", "teresa": "F", "thais": "F", "tinta":
    "F", "tomas": "M", "tomás": "M", "toñi": "F", "toño": "M", "unai": "M", "uriel": "M",
    "valentina": "F", "valentín": "M", "valeria": "F", "vane": "F", "vanesa": "F", "vanessa":
    "F", "vero": "F", "veronica": "F", "vicent": "M", "vicente": "M", "victor": "M", "victoria":
    "F", "violeta": "F", "virginia": "F", "vivian": "F", "viviana": "F", "víctor": "M",
    "walter": "M", "wendy": "F", "xavi": "M", "xavier": "M", "ximena": "F", "yamil": "M",
    "yayo": "M", "yessica": "F", "yolanda": "F", "álvaro": "M", "ángel": "M", "ángela": "F",
    "óscar": "M"
}


## 6. Limpieza

Limpieza. Una función por transformación, todas activables por separado.

Cada función recibe un DataFrame y devuelve uno nuevo, sin mutar el original y sin
depender de estado global. Así el runner puede aplicarlas de a una y medir su efecto
aislado (CLAUDE.md 3.4).

El orden del pipeline es el de CLAUDE.md 3.5: filtrar → outliers a nulo → imputar →
dummies. No es cosmético: imputar antes de mandar los outliers a nulo dejaría los
valores imposibles adentro, y hacer dummies antes de imputar crearía una columna por
cada valor sucio.

Nota sobre el alcance: acá sólo se limpia lo que llegó al dataset a través de las
opiniones. Los libros y lectores sin ninguna opinión ya desaparecieron en el merge y
sus nulos no son un problema nuestro.

*(desde `src/limpieza.py`)*


In [8]:
import numpy as np
import pandas as pd


DESCONOCIDO = "Desconocido"

# --------------------------------------------------------------------------------------
# Rangos, justificados con la distribución (ver src/correr_limpieza.py)
# --------------------------------------------------------------------------------------

# Un lector no opina antes de los 10 ni después de los 90. Los percentiles 0,5 y 99,5
# de la edad al opinar son 7 y 85, así que el rango deja afuera menos del 1% y no toca
# la masa central. El 1910 se trata aparte: no es un extremo de la distribución, es el
# valor mínimo del selector del formulario, y por eso se descarta aunque caiga dentro.
EDAD_MIN, EDAD_MAX = 10, 90
NACIMIENTO_CENTINELA = 1910

# La imprenta de tipos móviles es de mediados del siglo XV; el tope es el año más
# reciente que aparece en las opiniones. Fuera de ahí el año no es un dato, es basura.
EDICION_MIN = 1450

COL_PAIS = "region"
COL_GENERO = "genero_lector_imputado"


# --------------------------------------------------------------------------------------
# 1. Filtrar
# --------------------------------------------------------------------------------------

def filtrar_sin_target(df: pd.DataFrame) -> pd.DataFrame:
    """Descarta las opiniones sin rating: sin target no sirven ni para entrenar ni para medir."""
    return df[df[config.COL_RATING].notna()].copy()


def descartar_no_libros(df: pd.DataFrame) -> pd.DataFrame:
    """Descarta las opiniones cuyo `id_libro` no existe como libro en el catálogo.

    Se buscó un volumen relevante de revistas, agendas y cuadernos mal catalogados y
    no aparece: los registros problemáticos no son "otro tipo de cosa", son
    referencias a libros que no están en la tabla. Sin título, autor, género ni
    editorial, la fila no aporta ninguna predictora.

    Equivale a haber hecho el merge con `how="inner"`, pero acá queda medido en vez
    de decidido de antemano.
    """
    return df[df["titulo"].notna()].copy()


# --------------------------------------------------------------------------------------
# 2. Outliers a nulo (todavía no se imputa)
# --------------------------------------------------------------------------------------

def anio_nacimiento_a_nulo(df: pd.DataFrame) -> pd.DataFrame:
    """Manda a NaN los años de nacimiento imposibles. No se imputan: el año no se estima.

    La decisión se toma por lector, no por fila: el año de nacimiento es una propiedad
    de la persona, así que si su edad implícita es imposible en alguna de sus opiniones,
    el año declarado no es creíble en ninguna. Anularlo sólo en algunas filas dejaría
    al mismo lector con dos años de nacimiento distintos.
    """
    out = df.copy()
    anio_opinion = pd.to_datetime(out["fecha"], format="%d-%m-%Y").dt.year
    edad = anio_opinion - out["nacimiento"]

    fuera_de_rango = (edad < EDAD_MIN) | (edad > EDAD_MAX)
    lectores_sospechosos = set(out.loc[fuera_de_rango, "id_lector"].unique())

    invalido = (
        out["nacimiento"].eq(NACIMIENTO_CENTINELA)
        | out["id_lector"].isin(lectores_sospechosos)
    )
    out.loc[invalido, "nacimiento"] = np.nan
    return out


def anio_edicion_a_nulo(df: pd.DataFrame) -> pd.DataFrame:
    """Convierte `anio_edicion` a número y manda a NaN lo ilegible y lo imposible.

    La conversión es el grueso del trabajo: la columna viene como texto y hay 833
    opiniones con restos de campos corridos (' (200', ' crít', ' 01-0'). Mientras sea
    texto, el modelo no la puede usar; convertida, pasa a ser una predictora.
    """
    out = df.copy()
    anio = pd.to_numeric(out["anio_edicion"], errors="coerce")
    tope = pd.to_datetime(out["fecha"], format="%d-%m-%Y").dt.year.max()
    out["anio_edicion"] = anio.where(anio.between(EDICION_MIN, tope))
    return out


def edicion_posterior_a_nulo(df: pd.DataFrame) -> pd.DataFrame:
    """Manda a NaN el año de edición cuando es posterior a la fecha de la opinión.

    Son 28.825 opiniones (7,4%). No es un error de carga: el sitio guarda la última
    edición de su catálogo, no el ejemplar que leyó el lector. Como dato del libro que
    se leyó, entonces, es incorrecto en esas filas.

    Se prueba por separado del resto porque anular el 7,4% de una predictora es una
    decisión distinta a corregir basura: hay que medir si compensa.
    """
    out = df.copy()
    anio = pd.to_numeric(out["anio_edicion"], errors="coerce")
    anio_opinion = pd.to_datetime(out["fecha"], format="%d-%m-%Y").dt.year
    out["anio_edicion"] = anio.where(anio <= anio_opinion)
    return out


# --------------------------------------------------------------------------------------
# 3. Imputar
# --------------------------------------------------------------------------------------

# Ciudades de las tres comunidades con lengua cooficial. La agrupación existe porque
# hay editoriales que publican sólo en catalán, gallego o euskera, así que vivir ahí
# cambia la oferta de libros disponible.
CIUDADES_COOFICIALES = {
    # Cataluña
    "barcelona", "tarragona", "girona", "gerona", "lleida", "lerida", "sabadell",
    "badalona", "terrassa", "tarrasa", "hospitalet de llobregat", "l'hospitalet de llobregat",
    "mataro", "mataró", "reus", "manresa", "sant cugat del valles", "vic", "igualada",
    "granollers", "vilanova i la geltru", "cornella de llobregat", "sitges",
    # Galicia
    "a coruña", "la coruña", "vigo", "ourense", "orense", "pontevedra", "lugo",
    "santiago de compostela", "ferrol", "vilagarcia de arousa", "narón", "naron",
    # País Vasco y Navarra euskaldun
    "bilbao", "getxo", "san sebastian", "san sebastián", "donostia", "vitoria",
    "vitoria-gasteiz", "gasteiz", "barakaldo", "irun", "irún", "portugalete",
    "santurtzi", "basauri", "getaria", "eibar", "durango", "leioa", "erandio",
}

PAISES_SUDAMERICA = {
    "chile", "colombia", "peru", "perú", "uruguay", "venezuela", "ecuador", "bolivia",
    "paraguay", "brasil", "brazil", "guyana", "suriname",
}

NULOS_DISFRAZADOS = {"", "-", "--", "¿?", "?", "n/a", "na", ".", "sin especificar"}


def _partir_vive_en(valor) -> tuple[str | None, str | None]:
    """Separa 'ciudad - país'. Devuelve (ciudad, país), cualquiera puede faltar."""
    if pd.isna(valor):
        return None, None
    texto = str(valor).strip().strip("-").strip()
    if texto.lower() in NULOS_DISFRAZADOS:
        return None, None
    if " - " in texto:
        ciudad, _, pais = texto.partition(" - ")
        return ciudad.strip().lower() or None, pais.strip().lower() or None
    # Sin separador el valor es un país suelto ("España", "Argentina") o una ciudad
    # a la que le quedó colgando el guion ("Tenerife -", que ya vino sin país).
    return None, texto.lower() or None


def normalizar_vive_en(df: pd.DataFrame) -> pd.DataFrame:
    """Separa ciudad y país, y los agrupa en regiones con sentido para el problema.

    Los grupos no son geográficos sino de mercado editorial: lo que cambia la oferta
    de libros es la lengua de publicación y el catálogo local.

    `España (sin especificar)` es un grupo propio y no se reparte entre los otros dos:
    son 100.418 opiniones, el 26% del dataset, que dicen sólo "España". Meterlas en
    "castellanohablante" sería inventar que ninguna es de Barcelona o Bilbao; meterlas
    en "desconocido" sería tirar el país, que sí sabemos.
    """
    out = df.copy()
    partido = out["vive_en"].map(_partir_vive_en)
    ciudad = partido.str[0]
    pais = partido.str[1]

    # PROBADO Y DESCARTADO: rescatar las 2.812 opiniones que traen "Ciudad -" con el
    # país vacío ("Madrid -", "Zaragoza -"), mirando qué ciudades aparecen con España
    # en el resto del dataset. Corrige ~350 filas españolas que hoy caen en "Otros",
    # pero rompe 3.938: alcanza con que exista una sola fila "Mexico - España" para
    # que las 3.673 opiniones que dicen "Mexico" pasen a ser España, y lo mismo con
    # Ecuador (265), "Berlín -" (66) y "London -" (1). El remedio es diez veces peor
    # que la enfermedad, así que las huérfanas quedan en "Otros".

    region = pd.Series(DESCONOCIDO, index=out.index, dtype=object)
    es_espania = pais.isin(("españa", "espana", "spain"))

    region[es_espania] = "España (sin especificar)"
    region[es_espania & ciudad.notna()] = "España castellanohablante"
    region[es_espania & ciudad.isin(CIUDADES_COOFICIALES)] = "España lengua cooficial"
    region[pais.eq("argentina")] = "Argentina"
    region[pais.isin(PAISES_SUDAMERICA)] = "Resto de Sudamérica"
    region[pais.notna() & (region == DESCONOCIDO)] = "Otros"

    out[COL_PAIS] = region
    return out


def imputar_genero_lector(df: pd.DataFrame) -> pd.DataFrame:
    """Completa el género del lector desde el nombre de pila.

    `genero_lector` trae el literal "-" en el 32% de las opiniones: es un nulo
    disfrazado que pandas no detecta, así que primero hay que reconocerlo como nulo.

    Los ambiguos, los nombres que no son nombres (`giovanniro255`, `j2c2`, `141008`) y
    los que no están en el diccionario van a "Desconocido". No es una derrota: en el
    análisis exploratorio ese grupo mostró una tasa de gusto del 81% contra 86% y 84%
    de hombres y mujeres, así que la ausencia del dato es informativa por sí misma.
    """
    out = df.copy()
    declarado = out["genero_lector"].where(~out["genero_lector"].isin(["-"]))

    pila = out["nombre"].str.strip().str.lower().str.split().str[0]
    es_nombre = pila.notna() & ~pila.str.contains(r"\d", na=True) & pila.str.len().ge(2)
    imputado = pila.where(es_nombre).map(GENERO_POR_NOMBRE).map({"M": "Hombre", "F": "Mujer"})

    out[COL_GENERO] = declarado.fillna(imputado).fillna(DESCONOCIDO)
    return out


def imputar_categoricas(df: pd.DataFrame) -> pd.DataFrame:
    """Rellena las categóricas restantes con "Desconocido", no con la moda.

    Imputar con la moda inventa un dato: convierte "no sabemos la editorial" en
    "la editorial es Debolsillo", que es falso en la enorme mayoría de los casos y le
    da al modelo una señal que no existe. "Desconocido" es una categoría legítima y
    deja que el modelo decida si la ausencia significa algo.

    De paso normaliza mayúsculas y espacios: `genero_libro` tiene 62 categorías que
    son 53 ("No Ficción" y "No ficción", "HIstórica y aventuras"), y esa diferencia
    es tipeo, no información.
    """
    out = df.copy()
    for col in ["genero_libro", "editorial", "autor"]:
        normalizada = (out[col].str.strip().str.lower()
                       # Los acentos también separan categorías que son la misma:
                       # "clásicos" y "clasicos", "biografías" y el typo "biografiás".
                       # NFKD + descarte de los diacríticos lo resuelve con pandas y
                       # el codec ascii de la stdlib, sin `unidecode` (prohibida).
                       .str.normalize("NFKD")
                       .str.encode("ascii", "ignore").str.decode("ascii"))
        out[col] = normalizada.replace(list(NULOS_DISFRAZADOS), np.nan).fillna(DESCONOCIDO)
    return out


# --------------------------------------------------------------------------------------
# 4. Dummies
# --------------------------------------------------------------------------------------

# Sólo se hacen dummies de las categóricas de cardinalidad cerrada y chica. `autor`
# (19.551) y `editorial` (2.676) necesitan un top-N, y CLAUDE.md 3.3 exige que ese
# top-N se defina con value_counts() sobre train: va en la rama del split, no acá.
COLS_DUMMIES = [COL_PAIS, COL_GENERO, "genero_libro"]

# La guarda de idempotencia de `crear_dummies` compara por prefijo, así que si el
# nombre de una columna fuera prefijo de otra ("genero_lector" y
# "genero_lector_imputado"), la segunda nunca recibiría sus dummies. Hoy no pasa;
# esto lo deja explícito para que no pase en silencio si se agrega una columna.
assert not any(a != b and a.startswith(f"{b}_") for a in COLS_DUMMIES for b in COLS_DUMMIES), \
    "Un nombre de COLS_DUMMIES es prefijo de otro: la guarda por prefijo fallaría."


def crear_dummies(df: pd.DataFrame) -> pd.DataFrame:
    """Convierte las categóricas de baja cardinalidad en columnas numéricas.

    Sin este paso, todo el trabajo sobre región, género del lector y género literario
    es invisible para el modelo, que sólo mira columnas numéricas.
    """
    out = df.copy()
    # Idempotente a propósito: el runner de experimentos aplica dummies a cada etapa
    # y el pipeline las aplica de nuevo al final. Sin esta guarda, la segunda pasada
    # duplica cada columna en silencio y el modelo entrena con features repetidas.
    presentes = [c for c in COLS_DUMMIES
                 if c in out.columns
                 and not any(col.startswith(f"{c}_") for col in out.columns)]
    if not presentes:
        return out
    dummies = pd.get_dummies(out[presentes], prefix=presentes, dtype="int8")
    return pd.concat([out, dummies], axis=1)


# --------------------------------------------------------------------------------------
# El pipeline, legible de arriba abajo
# --------------------------------------------------------------------------------------
# El orden es el de CLAUDE.md 3.5. Los pasos que empeoraron la métrica quedan
# comentados con el motivo, no borrados: la función sigue disponible para volver a
# probarla cuando cambien las variables.

PIPELINE = [
    # Delta marginal 0.0000. Hoy no descarta ninguna fila porque no hay ratings nulos.
    # Queda como guarda: si en otra corrida aparecen, no tienen que entrar.
    filtrar_sin_target,

    # descartar_no_libros,
    #   Delta marginal -0.0023 (dentro del ruido, pero negativo). Descarta 666 filas
    #   cuyo libro no existe en el catálogo. Además contradice la decisión de unir con
    #   how="left": si se activa, el dataset pasa a ser el del inner. No queda.

    # Delta marginal -0.0030, dentro del ruido. Se mantiene igualmente: un lector de
    # 116 años no es un dato con el que el modelo deba decidir, y el 1910 es un
    # artefacto del formulario. Corrige el valor, no la métrica.
    anio_nacimiento_a_nulo,

    # Delta marginal -0.0043, dentro del ruido. Se mantiene porque es el paso que
    # convierte `anio_edicion` de texto a número: sin él la columna no existe para el
    # modelo, y las ramas siguientes la necesitan como materia prima (antigüedad del
    # libro, distancia entre edición y lectura).
    anio_edicion_a_nulo,

    # edicion_posterior_a_nulo,
    #   Delta marginal -0.0012. Anula el año de edición en el 7,4% de las opiniones a
    #   cambio de nada. El dato es conceptualmente incorrecto (el sitio guarda la
    #   última edición, no la leída), pero el modelo no mejora al sacarlo. No queda.

    # Delta marginal +0.0052: el único paso de limpieza que supera la banda de ruido.
    normalizar_vive_en,

    # Delta marginal +0.0042, apenas por debajo del umbral. Se mantiene: recupera el
    # género de 89.103 opiniones y deja "Desconocido" como categoría, que el análisis
    # exploratorio mostró que discrimina (81% de gusto contra 86% y 84%).
    imputar_genero_lector,

    # Delta marginal +0.0007. Se mantiene por parsimonia, no por métrica: colapsa las
    # categorías que sólo diferían en mayúsculas y acentos, y el modelo termina con 11
    # columnas menos para el mismo f1.
    imputar_categoricas,

    # Delta marginal +0.0155: el cambio más grande de toda la rama. Sin dummies, todo
    # el trabajo sobre categóricas es invisible para el modelo.
    crear_dummies,
]


def aplicar(df: pd.DataFrame, funciones=None) -> pd.DataFrame:
    """Aplica las funciones en orden. Sin argumento, corre el pipeline completo."""
    out = df
    for funcion in (PIPELINE if funciones is None else funciones):
        out = funcion(out)
    return out


## 7. Modelo congelado de evaluación

Modelo congelado de evaluación: el instrumento con el que se mide cada cambio.

La regla del curso es ceteris paribus (CLAUDE.md 3.4): un cambio por vez contra el
mismo modelo. Por eso los hiperparámetros están fijos en `config.MODELO_CONGELADO` y
no se tocan hasta la rama de optimización. Si el modelo cambiara entre experimentos,
la diferencia de métrica no mediría el cambio, mediría el modelo.

*(desde `src/modelo.py`)*


In [9]:
import time

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split


# Una sola métrica de decisión (CLAUDE.md 3.2). `accuracy` no está y no va a estar:
# con clases ~84/16, predecir siempre 1 da 84% y no significa nada. Por el mismo
# motivo la métrica se calcula sobre la clase `config.CLASE_MEDIDA`.
METRICAS = {
    "f1": f1_score,
    "precision": precision_score,
    "recall": recall_score,
}


def es_derivada_del_target(columna: str) -> bool:
    """¿La columna contiene el target y por lo tanto no puede ser predictora?

    `rating` es el target antes de binarizar y `gusto` es el target. Cualquier
    columna derivada de ellos (`rating_medio_del_lector`, `gusto_previo`, ...)
    tambien queda afuera: entrenar con ellas es fuga de información (CLAUDE.md 3.3).
    """
    nombre = columna.lower()
    return config.COL_RATING in nombre or config.TARGET in nombre


def separar_x_y(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Devuelve X (predictoras numéricas, sin nulos) e y (el target).

    Se queda sólo con las columnas numéricas y rellena con un centinela para que el
    modelo pueda correr aunque el dataset todavía esté sucio. Ni el descarte de las
    categóricas ni el relleno son decisiones de modelado: son el mínimo necesario
    para que el instrumento funcione sobre datos crudos. Convertir categóricas en
    dummies e imputar con criterio son cambios que se miden con esta misma función.
    """
    if config.TARGET not in df.columns:
        raise ValueError(f"El DataFrame no tiene la columna `{config.TARGET}`.")

    y = df[config.TARGET]
    predictoras = [c for c in df.columns if not es_derivada_del_target(c)]
    X = df[predictoras].select_dtypes(include="number").fillna(config.RELLENO_NULOS)

    if X.empty or X.shape[1] == 0:
        raise ValueError("No quedó ninguna columna numérica para entrenar.")
    return X, y


def modelo_congelado() -> RandomForestClassifier:
    """El clasificador de evaluación, siempre con los mismos hiperparámetros."""
    return RandomForestClassifier(**config.MODELO_CONGELADO)


def evaluar(df: pd.DataFrame, nombre_experimento: str) -> dict:
    """Entrena el modelo congelado sobre `df` y devuelve las métricas del experimento.

    La brecha (test − train) es tan importante como la métrica: si el test mejora
    pero la brecha se agranda, el cambio está sobreajustando y no generalizando.
    """
    inicio = time.perf_counter()

    X, y = separar_x_y(df)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=config.TEST_SIZE,
        random_state=config.SEED,
        stratify=y,  # las clases están ~84/16: sin estratificar el split las corre
    )

    modelo = modelo_congelado().fit(X_train, y_train)
    metrica = METRICAS[config.METRICA]
    m_train = metrica(y_train, modelo.predict(X_train), pos_label=config.CLASE_MEDIDA)
    m_test = metrica(y_test, modelo.predict(X_test), pos_label=config.CLASE_MEDIDA)

    # Las claves son genéricas (`metrica_*`) y no `f1_*`: cuál es la métrica se
    # declara una sola vez, en config.METRICA, y no se repite en cada columna.
    return {
        "cambio": nombre_experimento,
        "metrica": f"{config.METRICA} (clase {config.CLASE_MEDIDA})",
        "metrica_train": round(m_train, 4),
        "metrica_test": round(m_test, 4),
        "brecha": round(m_test - m_train, 4),
        "filas": len(X),
        "columnas": X.shape[1],
        "segundos": round(time.perf_counter() - inicio, 1),
    }


## 8. Registro de experimentos

Registro de experimentos: la tabla que la cátedra pide en el informe.

Cada fila es un cambio probado solo, contra el modelo congelado (CLAUDE.md 3.4).
La comparación siempre es contra el experimento #0, la base sin limpiar: eso es lo
que mide `delta`.

*(desde `src/experimentos.py`)*


In [10]:
import pandas as pd


COLUMNAS = ["#", "cambio", "metrica_train", "metrica_test", "brecha",
            "delta", "delta_marginal", "filas", "columnas", "segundos", "queda"]

BASE = "#0"  # etiqueta de la fila base en la columna `queda`
# Las filas de resumen empiezan con "=". No son un paso más: comparar su delta
# marginal contra la fila de arriba no significa nada, porque la de arriba puede ser
# una variante descartada y no el paso anterior del pipeline.
PREFIJO_RESUMEN = "="


def _veredicto(delta_marginal) -> str:
    """sí / no / ruido, según el delta marginal contra la banda medida del instrumento."""
    if pd.isna(delta_marginal):
        return ""
    if abs(delta_marginal) < config.UMBRAL_RUIDO:
        return "ruido"
    return "sí" if delta_marginal > 0 else "no"


def tabla() -> pd.DataFrame:
    """Devuelve la tabla de experimentos ordenada por número. Vacía si no existe."""
    if not config.TABLA_EXPERIMENTOS.is_file():
        return pd.DataFrame(columns=COLUMNAS)
    return pd.read_csv(config.TABLA_EXPERIMENTOS).sort_values("#").reset_index(drop=True)


def registrar(resultado: dict) -> pd.DataFrame:
    """Agrega (o actualiza) una fila en resultados/tabla_experimentos.csv.

    Si el experimento ya estaba registrado con el mismo nombre, se pisa la fila en
    lugar de duplicarla: volver a correr un experimento es normal, tener dos filas
    con el mismo nombre y números distintos no.

    Hay dos deltas y miden cosas distintas:

    - `delta` compara contra el #0 y responde "¿el pipeline hasta acá es mejor que la
      base?". Es acumulado.
    - `delta_marginal` compara contra el paso anterior y responde "¿este cambio, solo,
      aportó algo?". Es el que corresponde a ceteris paribus (CLAUDE.md 3.4), y el
      que decide `queda`.

    `queda` marca "ruido" cuando el delta marginal cae dentro de la banda medida del
    instrumento (config.UMBRAL_RUIDO): ahí no se puede afirmar ni que sirve ni que no.
    La decisión final es del informe, no de esta función.
    """
    previa = tabla()
    ya_estaba = previa["cambio"] == resultado["cambio"]

    numero = int(previa.loc[ya_estaba, "#"].iloc[0]) if ya_estaba.any() else len(previa)

    fila = {
        "#": numero,
        "cambio": resultado["cambio"],
        "metrica_train": resultado["metrica_train"],
        "metrica_test": resultado["metrica_test"],
        "brecha": resultado["brecha"],
        "delta": None,
        "delta_marginal": None,
        "filas": resultado["filas"],
        "columnas": resultado["columnas"],
        "segundos": resultado["segundos"],
        "queda": None,
    }

    nueva = previa[~ya_estaba] if ya_estaba.any() else previa
    nueva = pd.concat([nueva, pd.DataFrame([fila])], ignore_index=True).sort_values("#")

    # El delta se recalcula sobre toda la tabla: si se vuelve a correr la base,
    # todos los deltas que dependen de ella tienen que moverse con ella.
    base = nueva.loc[nueva["#"] == 0, "metrica_test"]
    if not base.empty:
        referencia = float(base.iloc[0])
        nueva["delta"] = (nueva["metrica_test"] - referencia).round(4)
        nueva["delta_marginal"] = nueva["metrica_test"].diff().round(4)
        nueva["queda"] = nueva["delta_marginal"].apply(_veredicto)
        nueva.loc[nueva["#"] == 0, ["delta_marginal", "queda"]] = [None, BASE]
        resumen = nueva["cambio"].str.startswith(PREFIJO_RESUMEN)
        nueva.loc[resumen, ["delta_marginal", "queda"]] = [None, "final"]

    config.DIR_RESULTADOS.mkdir(parents=True, exist_ok=True)
    nueva[COLUMNAS].to_csv(config.TABLA_EXPERIMENTOS, index=False)
    return nueva[COLUMNAS].reset_index(drop=True)


## 9. Experimento #0 — la base sin limpiar

Corre el experimento #0: la base sin limpiar, contra el modelo congelado.

Es el número de referencia contra el que se comparan todos los cambios de las
ramas siguientes.

Uso:  python -m src.correr_base

*(desde `src/correr_base.py`)*


In [11]:
import pandas as pd


NOMBRE_BASE = "#0 — Base sin limpiar"


def main_correr_base() -> pd.DataFrame:
    base = pd.read_pickle(config.CHECKPOINT_BASE)
    print(f"Dataset: {config.CHECKPOINT_BASE.name} — {len(base):,} filas × {base.shape[1]} columnas")

    X, _ = modelo.separar_x_y(base)
    descartadas = [c for c in base.columns if c not in X.columns]
    print(f"\nPredictoras que sobreviven ({X.shape[1]}): {list(X.columns)}")
    print(f"Descartadas ({len(descartadas)}): {descartadas}")
    print("  (las categóricas todavía no son dummies y `rating`/`gusto` son el target)")

    print(f"\nEntrenando el modelo congelado — métrica: {config.METRICA}")
    resultado = modelo.evaluar(base, NOMBRE_BASE)
    print(resultado)

    tabla = experimentos.registrar(resultado)
    print(f"\nTabla de experimentos → {config.TABLA_EXPERIMENTOS}")
    print(tabla.to_string(index=False))
    return tabla


## 10. Los experimentos de limpieza

Prueba las limpiezas de a una, en forma acumulativa, contra el modelo congelado.

Cada paso agrega UNA transformación sobre el anterior y produce una fila de la tabla
de experimentos con su delta contra el #0. Ceteris paribus (CLAUDE.md 3.4).

Detalle de método: antes de evaluar, cada etapa pasa por `crear_dummies`. El modelo
congelado sólo mira columnas numéricas, así que sin ese paso las transformaciones
sobre categóricas —región, género del lector, género literario— darían delta cero, no
porque no sirvan sino porque el instrumento no las ve. Las dummies acá son parte del
aparato de medición; en el pipeline final son el último paso, como manda CLAUDE.md 3.5.

Uso:  python -m src.correr_limpieza

*(desde `src/correr_limpieza.py`)*


In [12]:
import pandas as pd


# Cada entrada es (etiqueta, función). Se aplican de forma acumulativa.
# La fila de referencia separa el efecto de las dummies del de la primera limpieza.
# Sin ella, el paso 1 se llevaría el crédito de un salto que en realidad es de las
# dummies del género literario.
REFERENCIA = ("(referencia) sólo dummies, sin limpiar", lambda df: df)

PASOS = [
    ("filtrar opiniones sin rating", limpieza.filtrar_sin_target),
    ("descartar libros inexistentes", limpieza.descartar_no_libros),
    ("año de nacimiento imposible → nulo", limpieza.anio_nacimiento_a_nulo),
    ("año de edición ilegible → nulo", limpieza.anio_edicion_a_nulo),
    ("normalizar vive_en → región", limpieza.normalizar_vive_en),
    ("imputar género del lector", limpieza.imputar_genero_lector),
    ("imputar categóricas con Desconocido", limpieza.imputar_categoricas),
]

# Se prueba aparte, sobre el pipeline completo: anular el 7,4% de una predictora es
# una decisión distinta a corregir basura.
APARTE = [("año de edición posterior a la opinión → nulo", limpieza.edicion_posterior_a_nulo)]


def evaluar_etapa(df: pd.DataFrame, etiqueta: str) -> dict:
    """Aplica dummies y mide. Las dummies son del aparato de medición, no del paso."""
    return modelo.evaluar(limpieza.crear_dummies(df), etiqueta)


def main_correr_limpieza() -> pd.DataFrame:
    pd.set_option("display.width", 220)
    base = pd.read_pickle(config.CHECKPOINT_BASE)
    print(f"Base: {len(base):,} filas × {base.shape[1]} columnas\n")

    # La tabla se reconstruye entera: los deltas marginales dependen del orden y de
    # los vecinos, así que una tabla con filas de corridas distintas no sería coherente.
    config.TABLA_EXPERIMENTOS.unlink(missing_ok=True)

    print("#0 base sin limpiar (sin dummies: como la ve el modelo hoy)")
    experimentos.registrar(modelo.evaluar(base, "Base sin limpiar"))

    etiqueta, funcion = REFERENCIA
    print(f"#1 {etiqueta}")
    experimentos.registrar(evaluar_etapa(funcion(base), etiqueta))

    acumulado = base
    for numero, (etiqueta, funcion) in enumerate(PASOS, start=2):
        antes = len(acumulado)
        acumulado = funcion(acumulado)
        print(f"#{numero} +{etiqueta}  ({antes:,} → {len(acumulado):,} filas)")
        experimentos.registrar(evaluar_etapa(acumulado, f"+{etiqueta}"))

    numero = len(PASOS) + 2
    for etiqueta, funcion in APARTE:
        print(f"#{numero} +{etiqueta}  (aparte, sobre el pipeline completo)")
        experimentos.registrar(evaluar_etapa(funcion(acumulado), f"+{etiqueta}"))
        numero += 1

    # El checkpoint lleva SÓLO las transformaciones que quedaron en el PIPELINE.
    print("\nPipeline final (sólo lo que quedó):")
    for funcion in limpieza.PIPELINE:
        print(f"  · {funcion.__name__}")
    limpio = limpieza.aplicar(base)
    experimentos.registrar(modelo.evaluar(limpio, "= PIPELINE FINAL (sólo lo que quedó)"))
    config.DIR_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
    limpio.to_pickle(config.CHECKPOINT_LIMPIO)
    print(f"\nGuardado: {config.CHECKPOINT_LIMPIO} — "
          f"{limpio.shape[0]:,} filas × {limpio.shape[1]} columnas")

    tabla = experimentos.tabla()
    print("\n" + "=" * 110)
    print(f"TABLA DE EXPERIMENTOS — métrica: {config.METRICA} sobre la clase {config.CLASE_MEDIDA}")
    print("=" * 110)
    print(tabla.to_string(index=False))
    print(f"\nBanda de ruido del instrumento: ±{config.UMBRAL_RUIDO}. "
          f"`queda` se decide con el delta MARGINAL, no con el acumulado.")
    return tabla


## 11. Ejecución

Las tres etapas, en orden. Cada `main` lleva el nombre de su módulo porque en un
notebook todo comparte el mismo espacio de nombres.

`main_correr_limpieza` reconstruye la tabla de experimentos entera, incluida la fila
del #0, así que no hace falta llamar a `main_correr_base` por separado.


In [13]:
%%time
base = main_diagnostico()      # carga, une, construye el target y deja 01_base.pkl
main_eda()                     # las figuras del informe
tabla = main_correr_limpieza() # los experimentos de limpieza y 04_limpio.pkl



0. LECTURA DE LAS TRES TABLAS
libros:    /home/user/What_Book_Do_I_Read/libros.csv
lectores:  /home/user/What_Book_Do_I_Read/lectores.csv
opiniones: /home/user/What_Book_Do_I_Read/interacciones.csv



1. PERFIL DE CADA TABLA POR SEPARADO

### Tabla `libros` — 128,743 filas × 9 columnas
             tipo  % nulos  valores únicos
id_libro      str     0.00          128743
titulo        str    60.76           48356
autor         str    60.76           21654
genero        str    60.76              65
editorial     str    60.76            2908
anio_edicion  str    60.76             118
isbn          str    60.76           49930
resumen       str    62.10           47991
img_src       str    60.76           47861
  IDs únicos en `id_libro`: 128,743
  Filas duplicadas por la clave ['id_libro']: 0

### Tabla `lectores` — 11,285 filas × 5 columnas
               tipo  % nulos  valores únicos
id_lector       str     0.00           11285
nombre          str     0.00            8830
genero          str     0.00               3
vive_en         str     4.73            1584
nacimiento  float64    30.39              94
  IDs únicos en `id_lector`: 11,285
  Filas duplicadas por la clave ['id_lector

            tipo  % nulos  valores únicos
id_lector    str      0.0           10689
id_libro     str      0.0           50787
fecha        str      0.0            6619
rating     int64      0.0              10
  IDs únicos en `id_lector`: 10,689
  IDs únicos en `id_libro`: 50,787
  Filas duplicadas por la clave ['id_lector', 'id_libro']: 0

2. COBERTURA: CUÁNTO DE CADA TABLA USA EL DATASET


          en la tabla  con al menos una opinión  % usado  sin ninguna opinión  % descartado por el merge
tabla                                                                                                   
libros         128743                     50451    39.19                78292                      60.81
lectores        11285                     10683    94.67                  602                       5.33

Opiniones huérfanas (referencian un ID que no existe):
                                                opiniones  % del total
caso                                                                  
libro inexistente                                     815        0.170
lector inexistente                                    142        0.030
libro Y lector inexistentes                             0        0.000
al menos una de las dos (se pierden con inner)        957        0.199



3. NULOS ANTES Y DESPUÉS DEL MERGE


...

#7 +imputar género del lector  (388,842 → 388,842 filas)


#8 +imputar categóricas con Desconocido  (388,842 → 388,842 filas)


#9 +año de edición posterior a la opinión → nulo  (aparte, sobre el pipeline completo)



Pipeline final (sólo lo que quedó):
  · filtrar_sin_target
  · anio_nacimiento_a_nulo
  · anio_edicion_a_nulo
  · normalizar_vive_en
  · imputar_genero_lector
  · imputar_categoricas
  · crear_dummies



Guardado: /home/user/What_Book_Do_I_Read/checkpoints/04_limpio.pkl — 389,508 filas × 81 columnas

TABLA DE EXPERIMENTOS — métrica: f1 sobre la clase 0
 #                                        cambio  metrica_train  metrica_test  brecha  delta  delta_marginal  filas  columnas  segundos queda
 0                              Base sin limpiar         0.2903        0.2883 -0.0019 0.0000             NaN 389508         1       9.9    #0
 1        (referencia) sólo dummies, sin limpiar         0.3059        0.3038 -0.0021 0.0155          0.0155 389508        63      12.5    sí
 2                 +filtrar opiniones sin rating         0.3059        0.3038 -0.0021 0.0155          0.0000 389508        63      12.3 ruido
 3                +descartar libros inexistentes         0.3076        0.3015 -0.0061 0.0132         -0.0023 388842        63      11.8 ruido
 4           +año de nacimiento imposible → nulo         0.3044        0.2985 -0.0059 0.0102         -0.0030 388842        63      11.2 ru